In [39]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Model 
from tensorflow.keras.layers import LSTM,Dense,Input 


In [40]:
input_text=["hi","hello","how are you"]
target_text=["salut","bonjour","comment ca va"]

In [41]:
#adding start and end
target_text = ["\t"+txt+"\n" for txt in target_text]

In [46]:
#charector tockenisation

input_chars = sorted(set("".join(input_text)))
target_chars = sorted(set("".join(target_text)))

input_char_index = {char:i for i,char in enumerate(input_chars)}
target_char_index = {char:i for i,char in enumerate(target_chars)}

num_encoder_tokens = len(input_chars)
num_decoder_tokens = len(target_chars)

max_encoder_len = max(len(txt) for txt in input_text)
max_decoder_len = max(len(txt) for txt in target_text)

In [47]:
#vectorization

encoder_input_data = np.zeros(
    (len(input_text),max_encoder_len, num_encoder_tokens),
    dtype='float32')

decoder_input_data = np.zeros(
    (len(input_text),max_decoder_len, num_decoder_tokens),
    dtype='float32')

decoder_output_data = np.zeros(
    (len(input_text),max_decoder_len,num_decoder_tokens),
    dtype='float32')

In [48]:
for i,(input_text,target_text) in enumerate(zip(input_text,target_text)):
    for t,char in enumerate(input_text):
        encoder_input_data[i,t,input_char_index[char]]=1
    for t,char in enumerate(target_text):
        decoder_input_data[i,t,target_char_index[char]]=1
        if t>0:
            decoder_output_data[i,t-1,target_char_index[char]]=1

In [50]:
print("max_decoder_len =", max_decoder_len)
print("decoder_input_data.shape =", decoder_input_data.shape)
print("decoder_output_data.shape =", decoder_output_data.shape)

max_decoder_len = 1
decoder_input_data.shape = (2, 1, 7)
decoder_output_data.shape = (2, 1, 7)


In [51]:
#Build Encoder
latent_dim=64

encoder_inputs = Input(shape=(None,num_encoder_tokens))
encoder_lstm = LSTM(latent_dim,return_state=True)
encoder_outputs,state_h,state_c=encoder_lstm(encoder_inputs)

encoder_state = [state_h,state_c]

In [54]:
decoder_inputs = Input(shape=(None,num_decoder_tokens))
decoder_lstm = LSTM(latent_dim,return_sequences=True,return_state=True)

decoder_outputs,_,_ = decoder_lstm(
decoder_inputs,initial_state=encoder_state)

decoder_dense=Dense(num_decoder_tokens,activation='softmax')
decoder_outputs=decoder_dense(decoder_outputs)

In [55]:
#Create Model

model=Model([encoder_inputs,decoder_inputs],decoder_outputs)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy'
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, None, 2)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, None, 7)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, 64),      │     17,152 │ input_layer_1[0]… │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, None,     │     18,432 │ input_layer_2[0]… │
│                     │ 64), (None, 64),  │            │ lstm_1[0][1],     │
│                     │ (None, 64)]       │            │ lstm_1[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 7)   │        455 │ lstm_2[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 36,039 (140.78 KB)

 Trainable params: 36,039 (140.78 KB)

 Non-trainable params: 0 (0.00 B)

In [58]:
#Train Model

model.fit([encoder_input_data,decoder_input_data],
          decoder_output_data,
          batch_size=2,
          epochs=100)

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - loss: 0.0000e+00
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - loss: 0.0000e+00
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - loss: 0.0000e+00
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - loss: 0.0000e+00
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - loss: 0.0000e+00
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - loss: 0.0000e+00
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - loss: 0.0000e+00
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step - loss: 0.0000e+00
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - loss: 0.0000e+00
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - loss: 0.0000e+00
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - loss: 0.0000e+00
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step - loss: 0.0000e+00
Epoch 13/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - loss: 0.0000e+00
Epoch 14/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - loss: 0.0000e+00
Epoch 15/100
1/1 ━━━━━━━━━━━━